In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# # sample  =  pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# # sample.to_csv('submission.csv', index=False)
# !pip uninstall -y transformers sentence-transformers

# Smart MCQ Solver Challenge — End-to-End Solution
### MAP@3 ranking of top-3 answers (A–E) for knowledge-based MCQs

**Pipeline overview**

| # | Model | Category | Idea |
|---|-------|----------|------|
| 1 | TF-IDF + PyTorch MLP | **Built from scratch** | No pretrained weights anywhere; learns purely from the ~2,000 training rows |
| 2 | DeBERTa-v3 (`AutoModelForMultipleChoice`) | **Pretrained, fine-tuned** | Transfer learning — pretrained language + world knowledge, adapted to this task |
| 3 | XGBoost on engineered similarity features (TF-IDF sim, Sentence-Transformer sim, lexical overlap, etc.) | **Additional model of choice** | Tree-based, feature-driven — a structurally different failure mode from 1 & 2 |
| — | Weighted ensemble of the three | **Final submission** | Combines probability outputs, tuned on a held-out validation split |

**Notebook structure**
1. Setup & data loading
2. EDA (light)
3. MAP@3 metric implementation
4. Train/validation split
5. Model 1 — from-scratch TF-IDF + MLP
6. Model 2 — pretrained DeBERTa-v3 fine-tuned as multiple-choice classifier
7. Model 3 — XGBoost on engineered similarity features
8. Ensembling + local MAP@3 evaluation
9. Final inference on `test.csv` + submission file
10. (Optional, commented out) Zero-shot LLM prompting extension

> Upload `train.csv`, `test.csv`, and `sample_submission.csv` to the Colab working directory (or mount Google Drive) before running.


##1. Setup

In [3]:
# Run this once per Colab session.
# ============================================================
# KAGGLE: INSTALL ONLY WHAT YOU NEED (NO UPGRADES)
# ============================================================
# Install specific versions WITHOUT upgrading system packages
!pip install -q transformers==4.41.2 datasets==2.20.0 accelerate==0.31.0
!pip install -q sentence-transformers==2.7.0
!pip install -q scikit-learn==1.5.0 xgboost==2.0.3
!pip install -q wandb

# DO NOT upgrade numpy, torch, tensorflow, cuda, or RAPIDS packages

print("✅ Installation complete - using Kaggle's base packages")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 r

In [4]:
import os, re, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# this is now the errro , it is jsut a warning 


Using device: cuda


In [5]:
# ---- Paths: adjust if your files live elsewhere (e.g. Google Drive) ----
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge/"   # change to "/content/drive/MyDrive/mcq_challenge" if using Drive

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

OPTIONS = ["A", "B", "C", "D", "E"]
print(train.shape, test.shape, sample_submission.shape)
train.head()


(2000, 8) (500, 7) (500, 2)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## 2. Light EDA

In [6]:
print("Missing values (train):\n", train.isnull().sum())
print("\nAnswer class balance:\n", train["answer"].value_counts())
print("\nPrompt length stats (chars):\n", train["prompt"].str.len().describe())

Missing values (train):
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer class balance:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt length stats (chars):
 count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64


## 3. MAP@3 metric

For each question, if the true label appears at rank *k* (1-indexed) among our top-3 predictions,
the score for that question is `1/k`; if it doesn't appear in the top 3, the score is `0`.
The final metric is the mean over all questions.


In [7]:
def map_at_3(y_true, top3_preds):
    """
    y_true      : list/array of true labels, e.g. ['A','C',...]
    top3_preds  : list of lists, each an ordered top-3 prediction e.g. [['A','B','C'], ...]
    """
    scores = []
    for true_label, preds in zip(y_true, top3_preds):
        score = 0.0
        for rank, p in enumerate(preds[:3], start=1):
            if p == true_label:
                score = 1.0 / rank
                break
        scores.append(score)
    return float(np.mean(scores))

def probs_to_top3(prob_matrix, classes=OPTIONS):
    """prob_matrix: (n_samples, 5) array of class probabilities in the order of `classes`."""
    order = np.argsort(-prob_matrix, axis=1)  # descending
    top3 = [[classes[idx] for idx in row[:3]] for row in order]
    return top3


## 4. Train / validation split

We hold out 15% of the training data (stratified by answer label) purely for local MAP@3 evaluation
and ensemble-weight tuning. The final models are re-fit on the *full* training set before predicting on `test.csv`.


In [8]:
train_idx, val_idx = train_test_split(
    train.index, test_size=0.15, random_state=SEED, stratify=train["answer"]
)
tr_df  = train.loc[train_idx].reset_index(drop=True)
val_df = train.loc[val_idx].reset_index(drop=True)
print("train:", tr_df.shape, " val:", val_df.shape)

train: (1700, 8)  val: (300, 8)
